# Chapter 01: Why Actions? Cost vs Scripts and Jenkins (notebook edition)

## Learning Objectives

- Compute billed minutes by hand and in code
- See the rounding penalty
- Estimate monthly overage

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Per-job round-up

We bill Tally's three stages as three jobs and as one. You should see 3 minutes vs 1 minute.

In [ ]:
from intro_gha.cost import run_minutes, run_cost_mills, mills_to_dollars
SPLIT = [(15,"linux"),(20,"linux"),(25,"linux")]
MERGED = [(60,"linux")]
print(run_minutes(SPLIT), run_minutes(MERGED))
print(mills_to_dollars(run_cost_mills(SPLIT)), mills_to_dollars(run_cost_mills(MERGED)))
assert (run_minutes(SPLIT), run_minutes(MERGED)) == (3, 1)
assert (run_cost_mills(SPLIT), run_cost_mills(MERGED)) == (18, 6)

## 2. The monthly overage

At 1,000 pushes a month on the Free plan (2,000 included minutes), the split layout crosses the allowance.

In [ ]:
from intro_gha.cost import FREE_MINUTES
def overage_mills(jobs, pushes, plan):
    over = max(0, pushes * run_minutes(jobs) - FREE_MINUTES[plan])
    return over * 6   # Linux mills per minute
print(overage_mills(SPLIT, 200, "free"), overage_mills(SPLIT, 1000, "free"), overage_mills(MERGED, 1000, "free"))
assert overage_mills(SPLIT, 1000, "free") == 6000   # $6.00
assert overage_mills(MERGED, 1000, "free") == 0

## 3. The macOS surprise

The same 61 seconds costs 2 minutes everywhere; the rate is what differs.

In [ ]:
from intro_gha.cost import job_cost_mills
print(job_cost_mills(61,"linux"), job_cost_mills(61,"macos"))
assert job_cost_mills(61,"macos") == 124

## Takeaways & Next Steps

- Every job rounds up to a whole minute.
- Splitting buys parallelism at the price of rounded-up minutes.
- Next: Chapter 02, how a run actually flows from event to runner.

## Chapter Link

Read: `learning_modules/chapter_01_why_actions.md`